# Assignment 2 — Customer Churn Prediction using Logistic Regression

**Dataset:** Telco Customer Churn (Kaggle) — https://www.kaggle.com/datasets/blastchar/telco-customer-churn

**Objective:** Build a Logistic Regression model that predicts whether a telecom customer will churn, based on demographic and service-usage information.

> **Before running:** download `WA_Fn-UseC_-Telco-Customer-Churn.csv` from the Kaggle link above and place it in the same folder as this notebook.


## Task 1: Data Understanding (2 Marks)

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the dataset
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Shape of dataset:", df.shape)


In [ ]:
# 2. Display the first five records
df.head()


In [ ]:
# 3. Identify numerical, categorical, and target variables
print("Column data types:\n")
print(df.dtypes)


In [ ]:
# TotalCharges is often read as an object due to blank strings for new customers -
# convert it to numeric so it is treated as a numerical feature.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
# customerID is an identifier, not a real categorical feature - keep it aside
categorical_features = [c for c in df.select_dtypes(include=['object']).columns.tolist()
                         if c not in ['customerID', 'Churn']]
target_variable = 'Churn'

print("Numerical features:", numerical_features)
print("\nCategorical features:", categorical_features)
print("\nTarget variable:", target_variable)


## Task 2: Data Preprocessing (2 Marks)

In [ ]:
# Check for missing values
print("Missing values per column:\n")
print(df.isnull().sum()[df.isnull().sum() > 0])


In [ ]:
# Handle missing values
# TotalCharges has a small number of missing values (customers with tenure = 0,
# i.e. brand-new customers who have not been billed yet). We fill these with 0,
# since that reflects reality (they have not been charged anything yet).
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print("Remaining missing values:", df.isnull().sum().sum())


In [ ]:
# Drop the identifier column - it carries no predictive information
df = df.drop(columns=['customerID'])

# Encode the target variable: Yes -> 1, No -> 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Encode categorical features using one-hot encoding
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Shape after encoding:", df_encoded.shape)
df_encoded.head()


In [ ]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

# Split the dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


## Task 3: Model Development (3 Marks)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Scale numerical features - Logistic Regression converges better and faster
# on features that are on a similar scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Build and train the Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print("Model trained successfully.")


In [ ]:
# Predict customer churn on the test dataset
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

pd.DataFrame({'Actual': y_test.values[:10], 'Predicted': y_pred[:10]})


## Task 4: Model Evaluation (2 Marks)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)
import matplotlib.pyplot as plt

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print("\nFull classification report:\n")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(cmap='Blues', ax=ax, colorbar=False)
plt.title('Confusion Matrix - Logistic Regression')
plt.tight_layout()
plt.show()


In [ ]:
# Which features push predictions towards churn the most?
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print("Top 10 features increasing churn likelihood:")
print(coef_df.head(10))

print("\nTop 10 features decreasing churn likelihood:")
print(coef_df.tail(10))


### Observations

*(Fill these in after you run the notebook on the actual dataset - the numbers below are placeholders to show the kind of observation expected.)*

1. The model achieves a reasonable overall accuracy, but **recall on the churn class is noticeably lower than precision**, meaning the model misses a portion of customers who actually churn - a common issue on this dataset because non-churners outnumber churners (class imbalance).
2. Contract type, tenure, and monthly charges tend to be among the strongest predictors: customers on **month-to-month contracts with high monthly charges and low tenure** are most likely to churn, while customers with long-term contracts and add-on services (e.g., tech support, online security) are less likely to churn.
3. Because accuracy alone is misleading on an imbalanced target, precision/recall/F1 give a fuller picture of how well the model actually identifies at-risk customers - this is important for a business use case where missing a churner is costlier than a false alarm.


## Task 5: Conclusion (1 Mark)

*(Write your own 100-150 word conclusion here after running the notebook. A sample structure is given below - edit it with your actual results.)*

This project used Logistic Regression to predict customer churn for a telecom company based on demographic and service-usage data. After handling missing values, encoding categorical variables, and scaling numerical features, the model achieved an accuracy of **[insert value]**, with a precision of **[insert value]** and recall of **[insert value]** on the churn class. Key factors influencing churn included contract type, tenure, monthly charges, and the presence of support-related add-on services - customers with short tenure, month-to-month contracts, and higher monthly charges were more likely to churn. A key limitation of Logistic Regression here is that it assumes a **linear relationship between the features and the log-odds of churn**, so it may fail to capture more complex, non-linear interactions between features (e.g., how tenure and contract type jointly affect churn) that models like Random Forests or Gradient Boosting could capture better.
